In [ ]:
# Cell 1

import sys

# downgrade pip to version compatible with Meltano
!{sys.executable} -m pip install pip==25.1.1

# install required packages for BI analysis
!{sys.executable} -m pip install pandas matplotlib sqlalchemy pybigquery google-cloud-bigquery db-dtypes

In [ ]:
# Cell 2 - imports and display settings

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
# Cell 3 - config

# Update only if your project / dataset names differ
PROJECT_ID = "<your_gcp_project_id>"
DATASET = "<your_raw_dataset>"

# Optional: uncomment this if local auth is needed
# import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\path\to\your-service-account.json"

EXPORT_DIR = Path("analytics/exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ID:", PROJECT_ID)
print("DATASET:", DATASET)
print("EXPORT_DIR:", EXPORT_DIR.resolve())

In [ ]:
# Cell 4 - create BigQuery engine

CONNECTION_URI = f"bigquery://{PROJECT_ID}"
engine = create_engine(CONNECTION_URI)

print("BigQuery engine created successfully.")

In [ ]:
# Cell 5 - helper function

def run_query(sql: str) -> pd.DataFrame:
    """
    Run a SQL query against BigQuery and return a pandas DataFrame.

    Args:
        sql: SQL query string.

    Returns:
        pandas.DataFrame containing the query results.
    """
    return pd.read_sql(sql, engine)

In [ ]:
# Cell 6 - sanity check available mart tables

sql_tables = f"""
SELECT table_name
FROM `{PROJECT_ID}.{DATASET}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

tables_df = run_query(sql_tables)
tables_df

In [ ]:
# Cell 7 - monthly revenue trend query

sql_monthly_revenue = f"""
SELECT
    DATE_TRUNC(order_purchase_timestamp, MONTH) AS month,
    SUM(price) AS revenue
FROM `{PROJECT_ID}.{DATASET}.fct_order_items`
GROUP BY 1
ORDER BY 1
"""

monthly_revenue_df = run_query(sql_monthly_revenue)
monthly_revenue_df.head()

In [ ]:
# Cell 8 - monthly revenue trend chart

plt.figure(figsize=(12, 6))
plt.plot(monthly_revenue_df["month"], monthly_revenue_df["revenue"])
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 - top product categories by revenue

sql_revenue_by_category = f"""
SELECT
    COALESCE(p.product_category_name_english, 'unknown') AS product_category,
    SUM(f.price) AS revenue
FROM `{PROJECT_ID}.{DATASET}.fct_order_items` f
JOIN `{PROJECT_ID}.{DATASET}.dim_products` p
    ON f.product_id = p.product_id
GROUP BY 1
ORDER BY revenue DESC
LIMIT 10
"""

category_revenue_df = run_query(sql_revenue_by_category)
category_revenue_df

In [ ]:
# Cell 10 - top product categories by revenue chart

plot_df = category_revenue_df.sort_values("revenue")

plt.figure(figsize=(10, 6))
plt.barh(plot_df["product_category"], plot_df["revenue"])
plt.title("Top 10 Product Categories by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 - top sellers by revenue

sql_top_sellers = f"""
SELECT
    seller_id,
    SUM(price) AS revenue
FROM `{PROJECT_ID}.{DATASET}.fct_order_items`
GROUP BY seller_id
ORDER BY revenue DESC
LIMIT 10
"""

top_sellers_df = run_query(sql_top_sellers)
top_sellers_df

In [ ]:
# Cell 12 - top sellers by revenue chart

plot_df = top_sellers_df.sort_values("revenue")

plt.figure(figsize=(10, 6))
plt.barh(plot_df["seller_id"], plot_df["revenue"])
plt.title("Top 10 Sellers by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Seller ID")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11a - top sellers by revenue

sql_top_sellers = f"""
SELECT
    s.seller_city,
    s.seller_state,
    f.seller_id,
    SUM(f.price) AS revenue
FROM `{PROJECT_ID}.{DATASET}.fct_order_items` f
JOIN `{PROJECT_ID}.{DATASET}.dim_sellers` s
    ON f.seller_id = s.seller_id
GROUP BY
    s.seller_city,
    s.seller_state,
    f.seller_id
ORDER BY revenue DESC
LIMIT 10
"""

top_sellers_df = run_query(sql_top_sellers)
top_sellers_df

In [ ]:
top_sellers_df["seller_location"] = (
    top_sellers_df["seller_city"].str.title()
    + ", "
    + top_sellers_df["seller_state"]
)

In [ ]:
# Cell 12a - top sellers by revenue chart

plot_df = top_sellers_df.sort_values("revenue")

plt.figure(figsize=(10,6))
plt.barh(plot_df["seller_location"], plot_df["revenue"])
plt.title("Top 10 Sellers by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Seller Location")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 - average review score

sql_avg_review = f"""
SELECT
    AVG(review_score) AS avg_review_score
FROM `{PROJECT_ID}.{DATASET}.fct_reviews`
"""

avg_review_df = run_query(sql_avg_review)
avg_review_df

In [ ]:
# Cell 14 - order status distribution

sql_order_status = f"""
SELECT
    order_status,
    COUNT(*) AS orders
FROM `{PROJECT_ID}.{DATASET}.fct_orders`
GROUP BY 1
ORDER BY orders DESC
"""

order_status_df = run_query(sql_order_status)
order_status_df

In [ ]:
# Cell 15 - order status chart

plt.figure(figsize=(10, 5))
plt.bar(order_status_df["order_status"], order_status_df["orders"])
plt.title("Order Status Distribution")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16 - customer segmentation by purchase behavior

sql_customer_segmentation = f"""
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(price) AS total_spent
    FROM `{PROJECT_ID}.{DATASET}.fct_order_items`
    GROUP BY customer_id
)
SELECT
    customer_id,
    total_orders,
    total_spent,
    SAFE_DIVIDE(total_spent, total_orders) AS avg_order_value,
    CASE
        WHEN total_orders >= 10 AND total_spent >= 1000 THEN 'High Value Loyal'
        WHEN total_orders >= 5 THEN 'Repeat Customer'
        WHEN total_spent >= 500 THEN 'High Spend Occasional'
        ELSE 'Standard Customer'
    END AS customer_segment
FROM customer_orders
"""

customer_segmentation_df = run_query(sql_customer_segmentation)
customer_segmentation_df.head()

In [ ]:
# Cell 17 - customer segmentation summary

segment_summary_df = (
    customer_segmentation_df
    .groupby("customer_segment", as_index=False)
    .agg(
        customers=("customer_id", "count"),
        avg_total_spent=("total_spent", "mean"),
        avg_total_orders=("total_orders", "mean"),
    )
    .sort_values("customers", ascending=False)
)

segment_summary_df

In [ ]:
# Cell 17a - customer segmentation summary

segment_summary_df = (
    customer_segmentation_df
    .groupby("customer_segment", as_index=False)
    .agg(
        customers=("customer_id", "count"),
        avg_total_spent=("total_spent", "mean"),
        avg_total_orders=("total_orders", "mean"),
        avg_order_value=("avg_order_value", "mean"),
    )
    .sort_values("customers", ascending=False)
)

segment_summary_df

In [ ]:
# Cell 18 - customer segment chart

plt.figure(figsize=(10, 5))
plt.bar(segment_summary_df["customer_segment"], segment_summary_df["customers"])
plt.title("Customer Segments by Purchase Behavior")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 19 - real delivery performance analysis

sql_delivery_delay = f"""
SELECT
    DATE_TRUNC(order_purchase_timestamp, MONTH) AS month,
    AVG(DATE_DIFF(order_delivered_customer_date, order_purchase_timestamp, DAY)) AS avg_delivery_time_days,
    AVG(DATE_DIFF(order_delivered_customer_date, order_estimated_delivery_date, DAY)) AS avg_delivery_delay_vs_estimate
FROM `{PROJECT_ID}.{DATASET}.fct_orders`
WHERE order_purchase_timestamp IS NOT NULL
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

delivery_delay_df = run_query(sql_delivery_delay)
delivery_delay_df.head()

In [ ]:
# Cell 20 - delivery performance chart

plt.figure(figsize=(12, 6))
plt.plot(delivery_delay_df["month"], delivery_delay_df["avg_delivery_time_days"], label="Avg Delivery Time (days)")
plt.plot(delivery_delay_df["month"], delivery_delay_df["avg_delivery_delay_vs_estimate"], label="Avg Delay vs Estimate (days)")
plt.title("Delivery Performance by Month")
plt.xlabel("Month")
plt.ylabel("Days")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
delivery_delay_df = delivery_delay_df.rename(
    columns={
        "avg_delivery_delay_vs_estimate": "avg_days_early_or_late_vs_estimate"
    }
)

delivery_delay_df.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(
    delivery_delay_df["month"],
    delivery_delay_df["avg_delivery_time_days"],
    label="Avg Delivery Time (days)"
)
plt.plot(
    delivery_delay_df["month"],
    delivery_delay_df["avg_days_early_or_late_vs_estimate"],
    label="Avg Days Early/Late vs Estimate"
)

plt.title("Delivery Performance by Month")
plt.xlabel("Month")
plt.ylabel("Days")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 21 - optional KPI summary printout

top_category = category_revenue_df.iloc[0]["product_category"] if not category_revenue_df.empty else "N/A"
top_category_revenue = category_revenue_df.iloc[0]["revenue"] if not category_revenue_df.empty else 0

top_seller = top_sellers_df.iloc[0]["seller_id"] if not top_sellers_df.empty else "N/A"
top_seller_revenue = top_sellers_df.iloc[0]["revenue"] if not top_sellers_df.empty else 0

avg_review_score = avg_review_df.iloc[0]["avg_review_score"] if not avg_review_df.empty else 0

print("Executive KPI Summary")
print("-" * 40)
print(f"Top category by revenue : {top_category} ({top_category_revenue:,.2f})")
print(f"Top seller by revenue   : {top_seller} ({top_seller_revenue:,.2f})")
print(f"Average review score    : {avg_review_score:,.2f}")
print(f"Total segment groups    : {segment_summary_df.shape[0]}")
print(f"Monthly revenue rows    : {monthly_revenue_df.shape[0]}")

In [ ]:
# Cell 22 - export results to CSV

monthly_revenue_df.to_csv(EXPORT_DIR / "monthly_revenue.csv", index=False)
category_revenue_df.to_csv(EXPORT_DIR / "revenue_by_category.csv", index=False)
top_sellers_df.to_csv(EXPORT_DIR / "top_sellers.csv", index=False)
avg_review_df.to_csv(EXPORT_DIR / "average_review_score.csv", index=False)
order_status_df.to_csv(EXPORT_DIR / "order_status_distribution.csv", index=False)
customer_segmentation_df.to_csv(EXPORT_DIR / "customer_segmentation.csv", index=False)
segment_summary_df.to_csv(EXPORT_DIR / "customer_segmentation_summary.csv", index=False)
delivery_delay_df.to_csv(EXPORT_DIR / "delivery_performance_by_month.csv", index=False)

print("CSV exports created successfully.")
print("Saved to:", EXPORT_DIR.resolve())

In [ ]:
# Cell 23 - save chart images for README / slides

# 1) Monthly Revenue Trend
plt.figure(figsize=(12, 6))
plt.plot(monthly_revenue_df["month"], monthly_revenue_df["revenue"])
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "monthly_revenue_trend.png", dpi=150, bbox_inches="tight")
plt.show()

# 2) Top Product Categories by Revenue
plot_df = category_revenue_df.sort_values("revenue")
plt.figure(figsize=(10, 6))
plt.barh(plot_df["product_category"], plot_df["revenue"])
plt.title("Top 10 Product Categories by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product Category")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "top_categories.png", dpi=150, bbox_inches="tight")
plt.show()

# 3) Top Sellers by Revenue (seller location version)
plot_df = top_sellers_df.sort_values("revenue")
plt.figure(figsize=(10, 6))
plt.barh(plot_df["seller_location"], plot_df["revenue"])
plt.title("Top 10 Sellers by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Seller Location")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "top_sellers_by_location.png", dpi=150, bbox_inches="tight")
plt.show()

# 4) Order Status Distribution
plt.figure(figsize=(10, 5))
plt.bar(order_status_df["order_status"], order_status_df["orders"])
plt.title("Order Status Distribution")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "order_status_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# 5) Customer Segments
plt.figure(figsize=(10, 5))
plt.bar(segment_summary_df["customer_segment"], segment_summary_df["customers"])
plt.title("Customer Segments by Purchase Behavior")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "customer_segments.png", dpi=150, bbox_inches="tight")
plt.show()

# 6) Delivery Performance by Month
delay_col = (
    "avg_days_early_or_late_vs_estimate"
    if "avg_days_early_or_late_vs_estimate" in delivery_delay_df.columns
    else "avg_delivery_delay_vs_estimate"
)

delay_label = (
    "Avg Days Early/Late vs Estimate"
    if delay_col == "avg_days_early_or_late_vs_estimate"
    else "Avg Delay vs Estimate (days)"
)

plt.figure(figsize=(12, 6))
plt.plot(
    delivery_delay_df["month"],
    delivery_delay_df["avg_delivery_time_days"],
    label="Avg Delivery Time (days)"
)
plt.plot(
    delivery_delay_df["month"],
    delivery_delay_df[delay_col],
    label=delay_label
)
plt.title("Delivery Performance by Month")
plt.xlabel("Month")
plt.ylabel("Days")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(EXPORT_DIR / "delivery_performance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Chart images saved successfully.")
print("Saved to:", EXPORT_DIR.resolve())